### A. Co budujesz

Asystenta **agentowego**, nie klasycznego RAG-a. Klasyczny RAG to sztywno ustalony przebieg postępowania:

(`embed → pobierz → wklej kontekst → generuj`)

 — zawsze te same kroki, żadnej decyzji.

**Agent** sam decyduje,
czy, kiedy i którego narzędzia użyć, ogląda wynik i może wykonać kolejny krok.

### B. Proponowana zawężona tematyka PDF

Zbierz **3–5 PDF-ów na jeden, spójny temat** i wrzuć do folderu.

**Przepisy kulinarne** — trzeba nakarmić innych rozbitków. Przykładowe drugie narzędzie:
przeliczanie porcji, lista zakupów z kilku przepisów, porównanie z zawartością spiżarni, sumowanie czasu przygotowania.

**Instrukcje obsługi** (jakieś przedmioty przydatne do przetrwania). Przykładowe drugie narzędzie:
zużycie paliwa, przelicznik jednostek (PSI ↔ bar, °F ↔ °C), harmonogram serwisu, wyszukiwarka kodów błędów.

Drugie narzędzie ma mieć **weryfikowalne wyjście** — takie, po którym widać, że policzyło, a nie zgadło.
Narzędzie, które tylko przepisuje tekst albo którego agent nigdy nie wywoła, się nie liczy.

### C. Wybór narzędzi

| Warstwa | Czego najlepiej użyć |
|---|---
| Parsowanie PDF | PyMuPDF
| Podział na fragmenty | `RecursiveCharacterTextSplitter` |
| Embeddingi | model z HuggingFace |
| Baza wektorowa | Chroma lub Qdrant |
| Model generatywny | darmowe API z **natywnym tool callingiem** (Groq, Gemini) |
| Agent | `create_agent` z LangChain |
| Pamięć | `InMemorySaver` z LangGraph |

### 1. Środowisko i dane

In [1]:
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
!pip install -q langchain langchain-community langchain-text-splitters \
    langchain-huggingface langchain-chroma pymupdf \
    sentence-transformers langchain-google-genai #instalujemy potrzebne biblioteki
#!pip install langchain-groq

In [3]:
from langchain.agents import create_agent
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from datetime import datetime
from pathlib import Path
from google.colab import userdata


/tmp/ipykernel_5114/3547972379.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ścieżka pliku pdf
pdf_folder = "/content/drive/MyDrive/data/recipes"
pdf_files = [
    "recipes1.pdf",
    "recipes2.pdf",
    "recipes3.pdf"]
pages=[]

# ładowanie plików

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)

    loader = PyMuPDFLoader(file_path=pdf_path)
    pdf_pages = loader.load()

    print(f"Wczytano {pdf_file}: {len(pdf_pages)} stron")

    pages.extend(pdf_pages)

# podział na fragmenty
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

docs = text_splitter.split_documents(pages)
print("Liczba załadowanych stron:", len(pages))
print("Liczba utworzonych fragmentów:", len(docs))



Wczytano recipes1.pdf: 92 stron
Wczytano recipes2.pdf: 19 stron
Wczytano recipes3.pdf: 29 stron
Liczba załadowanych stron: 140
Liczba utworzonych fragmentów: 240


In [6]:
for idx,doc in enumerate(docs[75:79]):
  print(f"[{idx}].{doc}\n\n")


[0].page_content='Nutrition Facts Per Serving: Calories: 205 | Total Fat: 7 g | Saturated Fat: 1 g   
Sodium: 270 mg | Total Carbohydrate: 15 g | Protein: 23 g 
 
For more recipes, please visit www.nutrition.va.gov' metadata={'producer': '', 'creator': '', 'creationdate': '2022-04-05T07:55:20-07:00', 'source': '/content/drive/MyDrive/data/recipes/recipes1.pdf', 'file_path': '/content/drive/MyDrive/data/recipes/recipes1.pdf', 'total_pages': 92, 'format': 'PDF 1.7', 'title': '2022 Cooking Around the World Cookbook', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-24T15:11:00-04:00', 'trapped': '', 'modDate': "D:20260424151100-04'00'", 'creationDate': "D:20220405075520-07'00'", 'page': 43}


[1].page_content='40 
 
 
 Table of Contents 
         
Yellow Split Pea Soup 
Prep: 10 minutes | Cook: 2 hours  | Total:  2 hours 10 minutes 
Yield: 4 servings | Serving Size: ~1 cup  
Region: North America | Country: Canada 
Ingredients 
1 tablespoon unsalted butter  
1 medium carro

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
db_path = "/content/drive/MyDrive/data/recipes/chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=db_path
    )

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
    )

In [10]:
similar = retriever.invoke('What can we cook with potatoes?')
for doc in similar:
  print(doc.page_content[:300] + "...\n---")

2
DINING WITH DYSPHAGIA: A COOKBOOK
Start to Finish
45 minutes (20 active)
Servings
12
Ingredients
8 large russet potatoes  
(approximately 4 pounds)
1½ teaspoons salt, divided
¾ cup heavy whipping cream
¼ cup butter
½ teaspoon fresh rosemary
¼ teaspoon ground nutmeg
¼ teaspoon black pepper
Method
P...
---
27 
1. Preheat oven to 400 degrees and bake the potatoes or 1 hour or until done. When potatoes 
have cooked remove them from the oven to cool. 
2. As potatoes cool prepare soup by melting butter in a large saucepan, and sauté onion until 
light brown. Add the flour to the onions and stir to make a ...
---
69 
 
 
 Table of Contents 
         
Vegetable-Loaded Potato Salad 
Prep: 10 minutes | Cook: 20 minutes | Total: 30 minutes  
Yield: 4 servings | Serving Size: ~1 cup 
Region: Mediterranean Europe | Country: Portugal 
Ingredients 
Water 
1 teaspoon plus 1 pinch salt, divided 
2 medium (5- to 7-ounc...
---
17 
potato is sliced open, the inside of the skin will be charred black from

In [11]:
api_key = userdata.get('Gemini_API_Key')

os.environ["GOOGLE_API_KEY"] = api_key

In [12]:
@tool
def search_recipes(query: str) -> str:
    """
    Search the PDF knowledge base for recipe details.
    Use this for any questions about recipes, ingredients, cooking instructions, or preparation time.
    Answer based strictly only on the PDF documents. Always provide the source of the information.
    """
    print("DEBUG: search_recipes used")
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information was found in the recipe knowledge base."

    results = []

    for doc in docs:
        source = Path(doc.metadata["source"]).name
        page = doc.metadata.get("page", 0) + 1

        results.append(
            f"Source: {source}, page: {page}\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(results)

@tool
def ingredient_converter(
    quantity: float,
    original_servings: int,
    new_servings: int
) -> float:
    """
    Calculate a new ingredient quantity when changing the number of servings.
    Use this tool whenever the user asks to scale an ingredient.
    Use this tool for all quantities, including item counts.
    You can only do the calculations.
    """

    print("DEBUG: ingredient_converter used")

    if original_servings <= 0 or new_servings <= 0:
        raise ValueError("Number of servings must be greater than zero.")

    return quantity / original_servings * new_servings

print(
    search_recipes.invoke({
        "query": "recipe with potatoes"
    })
)

result = ingredient_converter.invoke({
    "quantity": 200,
    "original_servings": 4,
    "new_servings": 6
})

print("New quantity:", result)
tools = [search_recipes, ingredient_converter]

DEBUG: search_recipes used
Source: recipes2.pdf, page: 3
2
DINING WITH DYSPHAGIA: A COOKBOOK
Start to Finish
45 minutes (20 active)
Servings
12
Ingredients
8 large russet potatoes  
(approximately 4 pounds)
1½ teaspoons salt, divided
¾ cup heavy whipping cream
¼ cup butter
½ teaspoon fresh rosemary
¼ teaspoon ground nutmeg
¼ teaspoon black pepper
Method
Prepare ingredients: Wash, peel, and quarter the 
potatoes. Mince the rosemary leaves. Cube the butter 
and set aside to bring it down to room temperature.
1.	 Place potatoes in a large pot and cover with water. 
Add 1 teaspoon of salt and bring to a boil.
2.	 Reduce heat. Cover and simmer the potatoes for  
15-20 minutes until fork-tender. Drain.
3.	 Place potatoes in a large bowl. Add cream, butter, 
rosemary, nutmeg, pepper, and remaining salt. Use 
an electric mixer to beat until smooth and creamy.
4.	Divide the potatoes between bowls and serve  
with additional butter and a few sprigs of rosemary 
if desired.
Rosemary Mashed Potato

In [13]:
def parse_result(res):
  to_parse = res["messages"][-1]
  return to_parse.content[0]['text']


In [14]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt="""
You are a helpful culinary assistant on a desert island. Survivors of the Titanic disaster need to be fed so always refer to people as "survivors".
Answer questions strictly based on the provided PDF recipe base and available tools.
Always use 'search_recipes' for questions about recipes or ingredients. When the user asks for a recipe, provide the complete recipe found in the PDF.
If the recipe is incomplete do not include it in your answer.
If information is missing from the documents, clearly state that it was not found.
Use 'ingredient_converter' whenever ingredient amounts need to be adjusted for a different number of servings.
""",
    checkpointer=InMemorySaver()
    )

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

def run_agent(query: str, config: RunnableConfig):
  result = agent.invoke({"messages": [{"role": "user", "content": query}]}, config)
  return parse_result(result)

result_list = list()

result_list.append(run_agent("Give me a recipe that includes potatoes", config))
result_list.append(run_agent("How many ingredients are needed for 8 servings according to the 'Rosemary Mashed Potatoes' recipe?", config))
result_list.append(run_agent("I have 4 strawberries, what can I do with them", config))
result_list.append(run_agent("What day of the week is today?", config))
result_list.append(run_agent("What did I ask you in the beginning?", config))

for res in result_list:
  print(res)

DEBUG: search_recipes used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: search_recipes used
For you and the other survivors, here are a few recipes featuring potatoes found in the documents:

### **Rosemary Mashed Potatoes**
*Source: recipes2.pdf, page 3*
**Prep:** 20 minutes | **Cook:** 25 minutes | **Total:** 45 minutes
**Yield:** 12 servings

**Ingredients:**
*   8 large russet potatoes (approximately 4 pounds)
*   1½ teaspoons salt, divided
*   ¾ cup heavy whipping cream
*   ¼ cup butter
*   ½ teaspoon fresh rosemary
*   ¼ teaspoon ground nutmeg
*   ¼ teaspoon black pepper

**Method:**
1.  **Prepare ingredients:** Wash, peel, and quarter the potatoes. Mince the rosemary leaves. Cube the butter and set aside to bring it down to room temperature.
2.  Place potatoes in a large pot and cover with

Testy pokazują, że agent sam decyduje, którego narzędzia użyć.
Agent wykorzystuje `search_recipes`, by odpowiedzieć na pytania dotyczące przepisów czy składników, zaś w pytaniu o zmianę liczby porcji agent korzysta z `ingredient_converter`, aby wykonać obliczenie.
W przypadku pytań dotyczących informacji, których nie ma w bazie PDF ani na pytania wymagające danych w czasie rzeczywistym, takich jak data, czy aktualny czas, agent wyraźnie informuje, że nie może odpowiedzieć na to pytanie.
Ostatnie pytanie pokazuje, że pamięć agenta działa poprawnie i potrafi on odwołać się do wcześniejszych pytań użytkownika.